# LABEL — Verifikasi Label Cluster (Contact-Sheet + Merge/Split)

**Alat labeling manual untuk ground-truth identitas.** Skripsi FotoQu · Ade · TI UIN Alauddin Makassar

## Tujuan
Mengubah **proxy label** hasil HDBSCAN NB05 (144 cluster, otomatis) menjadi
**label identitas terverifikasi manusia**, yang dibutuhkan untuk:
- **Cluster Purity / ARI / NMI** (Skenario C) — butuh ground-truth per wajah.
- Kalibrasi θ (Skenario B) — subset berlabel untuk uji Mann-Whitney U.

## Prinsip: verifikasi, bukan label-dari-nol
Kamu **tidak** melabeli 12.000 wajah satu-satu. Kamu **memverifikasi** 144 cluster
yang sudah dibuat HDBSCAN. Untuk tiap cluster, cukup jawab: *"Apakah semua wajah di sini
satu orang?"* Empat kemungkinan keputusan:

| Keputusan | Arti | Fungsi |
|---|---|---|
| **OK** | cluster bersih, satu orang | `decide_ok()` atau `decide_ok('Nama')` |
| **MERGE** | dua cluster ternyata orang yang sama | `decide_merge(cluster_lain)` |
| **SPLIT** | satu cluster berisi ≥2 orang | `decide_split(k)` → `assign_split({...})` |
| **DISCARD** | bukan wajah / terlalu campur | `decide_discard()` |

## Strategi: 30 cluster terbesar dulu, sisanya bisa dilanjutkan kapan saja
Cluster diurutkan **besar → kecil**. 30 terbesar mencakup mayoritas instance wajah
(distribusi cluster foto dokumentasi organisasi umumnya *long-tail*). Progres disimpan
ke Drive **setiap keputusan** (`label_progress.json`) → berhenti kapan saja, buka lagi,
lanjut dari cluster berikutnya. **Tidak ada kerja yang hilang saat kernel Colab putus.**

## Justifikasi ilmiah (aturan CLAUDE.md #1, #4)
- **Otoritas label = mata manusia pada piksel wajah**, bukan geometri embedding.
  Sub-clustering saat SPLIT hanya **menata ulang wajah untuk inspeksi visual** — keputusan
  identitas tetap manusia. Ini memutus sirkularitas *"pakai clustering untuk memvalidasi
  clustering"*: ground-truth berasal dari verifikasi visual citra mentah.
- **Urutan tampil = jarak ke medoid** (cosine). Wajah "asing" (outlier, kandidat orang
  lain) muncul menonjol dengan bingkai merah → mempercepat deteksi cluster campur.
- **Keterbatasan (dicatat jujur):** seed label berasal dari HDBSCAN (satu keluarga dengan
  metode yang dievaluasi). Koreksi manusia (split/merge/discard) yang membuat label cukup
  independen. Purity/ARI mengukur kesesuaian clustering dengan penilaian manusia.
- **Compute:** seluruhnya **CPU**. Medoid & jarak O(n²·d) per cluster (n = ukuran cluster,
  puluhan–ratusan → trivial). Tidak butuh GPU. Tidak butuh InsightFace (pakai `bbox`
  tersimpan, bukan deteksi ulang).

## 0. Setup — dependencies & mount Drive
Ringan, CPU. Tanpa InsightFace/torch (tidak deteksi ulang).

In [ ]:
!pip install -q opencv-python-headless pillow pillow-heif rawpy scikit-learn pandas matplotlib tqdm ipywidgets
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Siap.")

## 1. Konfigurasi
Path harus sama dengan pipeline NB01/NB05. `REVIEW_FIRST` = jumlah cluster terbesar untuk
batch pertama (sisanya tetap bisa dilanjutkan). Parameter grid & outlier hanya alat bantu
visual, tidak memengaruhi label akhir.

In [ ]:
from pathlib import Path

# ── path (samakan dengan NB01/NB05) ──
BASE     = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings")
NB01_DIR = BASE / "output_nb01"      # embeddings.npy
NB05_DIR = BASE / "output_nb05"      # metadata_labeled.pkl (photo_path, bbox, cluster_id)
OUT_DIR  = BASE / "output_labeling"  # progres + hasil verifikasi
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── strategi review ──
REVIEW_FIRST = 30        # batch pertama: 30 cluster terbesar (sisanya lanjut kapan saja)

# ── parameter tampilan (alat bantu visual saja) ──
CAP_GRID  = 48           # maks wajah ditampilkan per cluster (sampling bila lebih)
GRID_COLS = 8            # kolom grid contact-sheet
OUTLIER_K = 1.5          # flag merah bila jarak-ke-medoid > mean + K·std
CROP_PAD  = 0.20         # padding bbox saat crop wajah (biar konteks kepala terlihat)

PROGRESS_PATH = OUT_DIR / "label_progress.json"
print("Config OK. Progres →", PROGRESS_PATH)

## 2. Muat data (metadata NB05 + embedding NB01)
`metadata_labeled.pkl` sudah memuat `photo_path`, `bbox`, `cluster_id` (self-consistent).
Embedding dipakai untuk urutan medoid & sub-clustering saat SPLIT.

In [ ]:
import pickle, numpy as np
from collections import Counter

with open(NB05_DIR / "metadata_labeled.pkl", "rb") as f:
    meta = pickle.load(f)
embeddings = np.load(NB01_DIR / "embeddings.npy").astype(np.float32)
assert len(meta) == len(embeddings), (len(meta), len(embeddings))

# L2-normalize (ArcFace) → cosine ≡ euclidean
embeddings /= (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-9)

labels = np.array([m["cluster_id"] for m in meta])
cluster_faces = {}                       # cid -> list index global
for i, c in enumerate(labels):
    if c < 0:   # noise HDBSCAN, tidak direview
        continue
    cluster_faces.setdefault(int(c), []).append(i)

sizes = Counter({c: len(idx) for c, idx in cluster_faces.items()})
review_queue = [c for c, _ in sizes.most_common()]   # besar → kecil
n_noise = int((labels < 0).sum())

print(f"Total wajah : {len(meta)}")
print(f"Cluster     : {len(cluster_faces)} (noise HDBSCAN: {n_noise} wajah, di-skip)")
print(f"Antrian review (besar→kecil), 5 teratas: "
      f"{[(c, sizes[c]) for c in review_queue[:5]]}")
print(f"Batch pertama: {REVIEW_FIRST} cluster terbesar "
      f"(≈ {sum(sizes[c] for c in review_queue[:REVIEW_FIRST])} wajah, "
      f"{100*sum(sizes[c] for c in review_queue[:REVIEW_FIRST])/max(1,len(meta)-n_noise):.0f}% dari wajah ter-cluster)")

## 3. Helper — baca gambar, crop, medoid, render grid
`imread_any` menangani JPG/HEIC/RAW. Cache foto (LRU) agar wajah sekali-foto tidak
di-decode berulang. Medoid = titik dengan total jarak minimum (robust terhadap outlier
dibanding mean).

In [ ]:
import cv2, math
from collections import OrderedDict
import matplotlib.pyplot as plt
from pillow_heif import register_heif_opener; register_heif_opener()

_PHOTO_CACHE = OrderedDict()
_CACHE_CAP = 64
def imread_any(p):
    if p in _PHOTO_CACHE:
        _PHOTO_CACHE.move_to_end(p); return _PHOTO_CACHE[p]
    img = cv2.imread(p)
    if img is None:
        ext = Path(p).suffix.lower()
        try:
            if ext in {'.raf','.cr2','.nef','.arw','.dng','.rw2','.orf'}:
                import rawpy
                with rawpy.imread(p) as raw:
                    img = cv2.cvtColor(raw.postprocess(), cv2.COLOR_RGB2BGR)
            else:
                from PIL import Image
                img = cv2.cvtColor(np.array(Image.open(p).convert("RGB")), cv2.COLOR_RGB2BGR)
        except Exception:
            img = None
    _PHOTO_CACHE[p] = img
    if len(_PHOTO_CACHE) > _CACHE_CAP: _PHOTO_CACHE.popitem(last=False)
    return img

def thumb(gi, size=112):
    m = meta[gi]; img = imread_any(m["photo_path"])
    if img is None: return np.zeros((size, size, 3), np.uint8)
    H, W = img.shape[:2]
    x1, y1, x2, y2 = m["bbox"]
    bw, bh = x2 - x1, y2 - y1
    x1 = int(max(0, x1 - CROP_PAD*bw)); y1 = int(max(0, y1 - CROP_PAD*bh))
    x2 = int(min(W, x2 + CROP_PAD*bw)); y2 = int(min(H, y2 + CROP_PAD*bh))
    crop = img[y1:y2, x1:x2]
    if crop.size == 0: return np.zeros((size, size, 3), np.uint8)
    return cv2.cvtColor(cv2.resize(crop, (size, size)), cv2.COLOR_BGR2RGB)

def medoid_dists(idx):
    X = embeddings[idx]                 # (n, 512) L2-normed
    dist = 1.0 - (X @ X.T)              # cosine distance ∈ [0,2]
    med = int(dist.sum(1).argmin())
    return med, dist[med]              # jarak tiap wajah ke medoid

def render_faces(idx_list, title="", flags=None, ranks=None, cols=None, cap=None):
    cols = cols or GRID_COLS; cap = cap or CAP_GRID
    show = idx_list[:cap]; n = len(show)
    if n == 0: print("(kosong)"); return
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols*1.5, rows*1.7))
    axes = np.array(axes).reshape(-1)
    for a in axes: a.axis("off")
    for j, gi in enumerate(show):
        ax = axes[j]; ax.imshow(thumb(gi))
        lbl = str(ranks[j]) if ranks is not None else str(j)
        red = flags is not None and flags[j]
        ax.set_title(lbl, fontsize=7, color=("red" if red else "black"))
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(bool(red)); s.set_edgecolor("red"); s.set_linewidth(2)
    if len(idx_list) > cap: title += f"  [tampil {cap}/{len(idx_list)}]"
    fig.suptitle(title, fontsize=12); plt.tight_layout(); plt.show()

## 4. Persistence progres (resumable) + union-find merge
Setiap keputusan disimpan ke `label_progress.json`. Aman terhadap kernel putus.

In [ ]:
import json
from datetime import datetime

def load_progress():
    if PROGRESS_PATH.exists():
        with open(PROGRESS_PATH) as f: return json.load(f)
    return {}

def save_progress():
    tmp = PROGRESS_PATH.with_suffix(".json.tmp")
    with open(tmp, "w") as f: json.dump(progress, f, ensure_ascii=False, indent=1)
    os.replace(tmp, PROGRESS_PATH)

progress = load_progress()
STATE = {"cursor": None, "pending_split": None}
print(f"Progres dimuat: {len(progress)} cluster sudah diputuskan.")

## 5. Contact-sheet + API keputusan
`review_next()` menampilkan cluster berikutnya yang belum diputuskan (urut besar→kecil).
Setiap `decide_*` menyimpan keputusan lalu **otomatis maju** ke cluster berikutnya.
Angka di atas tiap wajah = peringkat jarak ke medoid (0 = paling representatif). **Bingkai
merah = outlier** (kandidat orang lain / salah deteksi).

In [ ]:
def show_cluster(cid, decided_banner=True):
    cid = int(cid)
    idx = cluster_faces.get(cid)
    if not idx: print(f"Cluster {cid} tidak ada / kosong."); return
    med, d = medoid_dists(idx)
    order = np.argsort(d)                          # bersih → outlier
    idx_s = [idx[j] for j in order]; d_s = d[order]
    thr = d.mean() + OUTLIER_K*d.std()
    flags = [bool(v > thr) for v in d_s]; ranks = list(range(len(idx_s)))
    # sampling bila besar: separuh terdekat + separuh terjauh (surface kontaminasi)
    if len(idx_s) > CAP_GRID:
        h = CAP_GRID//2
        pick = list(range(h)) + list(range(len(idx_s)-h, len(idx_s)))
        idx_show = [idx_s[j] for j in pick]; flags_show=[flags[j] for j in pick]; ranks_show=[ranks[j] for j in pick]
    else:
        idx_show, flags_show, ranks_show = idx_s, flags, ranks
    prev = progress.get(str(cid)); st = prev["status"] if prev else "pending"
    ttl = f"Cluster {cid}  |  n={len(idx)}  |  jarak-medoid rata2={d.mean():.3f}  |  status={st}"
    render_faces(idx_show, ttl, flags_show, ranks_show)
    print(f"Outlier (d>μ+{OUTLIER_K}σ): {sum(flags)}/{len(idx)}. "
          f"Banyak & jelas beda orang → decide_split(). "
          f"Mirip cluster lain → decide_merge(cid_lain).")

def _next_pending(limit=None):
    q = review_queue if limit is None else review_queue[:limit]
    for c in q:
        if str(c) not in progress: return c
    return None

def review_next(limit=None):
    c = _next_pending(limit)
    if c is None:
        scope = "batch" if limit else "antrian"
        print(f"✓ Semua cluster dalam {scope} sudah direview. "
              f"(total diputuskan: {len(progress)}/{len(review_queue)})"); STATE["cursor"]=None; return
    STATE["cursor"] = int(c); STATE["pending_split"] = None
    pos = review_queue.index(c) + 1
    print(f"── Review cluster {c}  (urutan besar ke-{pos}/{len(review_queue)}) ──")
    show_cluster(c)

def _record(cid, status, **kw):
    rec = {"cluster_id": int(cid), "status": status, "ts": datetime.now().isoformat()}
    rec.update(kw); progress[str(cid)] = rec; save_progress()

def decide_ok(name=None, note=""):
    c = STATE["cursor"]; assert c is not None, "Jalankan review_next() dulu."
    tid = name or f"ID_{c:03d}"
    _record(c, "ok", true_id=tid, note=note); print(f"[OK] cluster {c} → {tid}"); review_next()

def decide_discard(note=""):
    c = STATE["cursor"]; assert c is not None
    _record(c, "discard", note=note); print(f"[DISCARD] cluster {c}"); review_next()

def decide_merge(into_cluster, name=None, note=""):
    c = STATE["cursor"]; assert c is not None
    _record(c, "merge", merge_with=int(into_cluster), true_id=name, note=note)
    print(f"[MERGE] cluster {c} → gabung ke {into_cluster}"); review_next()

## 6. Mekanisme SPLIT (cluster berisi ≥2 orang)
`decide_split(k)` menjalankan **Agglomerative (Ward)** pada embedding cluster → membaginya
jadi `k` sub-grup, lalu menampilkan tiap sub-grup terpisah. Ini **hanya proposal visual**;
kamu menetapkan identitas via `assign_split({...})`.

- `assign_split({0:'Budi', 1:'Ani'})` → sub-grup 0 = Budi, sub-grup 1 = Ani (identitas baru)
- nilai `'x'` atau `'discard'` → sub-grup itu dibuang
- nama identik dengan cluster/ sub-grup lain → otomatis tergabung (join by name)
- kurang bersih? ulangi `decide_split(3)` untuk k lebih besar.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

def propose_split(cid, k=2):
    idx = cluster_faces[int(cid)]; X = embeddings[idx]
    try:
        lab = AgglomerativeClustering(n_clusters=k, metric="euclidean", linkage="ward").fit_predict(X)
    except TypeError:  # sklearn lama
        lab = AgglomerativeClustering(n_clusters=k, affinity="euclidean", linkage="ward").fit_predict(X)
    for s in range(k):
        sub = [idx[j] for j in range(len(idx)) if lab[j] == s]
        render_faces(sub, f"Cluster {cid} — SUB-GRUP {s}  (n={len(sub)})")
    return lab, idx

def decide_split(k=2, note=""):
    c = STATE["cursor"]; assert c is not None
    lab, idx = propose_split(c, k)
    STATE["pending_split"] = {"cid": int(c), "lab": [int(x) for x in lab], "idx": [int(x) for x in idx],
                              "k": int(k), "note": note}
    print(f"Sub-grup k={k} untuk cluster {c}. Tetapkan identitas dengan:")
    print(f"  assign_split({{" + ", ".join(f"{s}:'nama'" for s in range(k)) + "}})   # 'x' = buang")

def assign_split(mapping):
    ps = STATE["pending_split"]; assert ps, "Jalankan decide_split() dulu."
    cid, lab, idx = ps["cid"], ps["lab"], ps["idx"]
    face_assign = {}
    for gi, s in zip(idx, lab):
        v = mapping.get(int(s), "x")
        face_assign[str(gi)] = None if str(v).lower() in ("x", "discard") else str(v)
    _record(cid, "split", k=ps["k"], note=ps["note"], split_assign=face_assign)
    kept = sum(v is not None for v in face_assign.values())
    print(f"[SPLIT] cluster {cid} → {ps['k']} sub-grup, {kept}/{len(idx)} wajah diberi identitas.")
    STATE["pending_split"] = None; review_next()

## 7. (Opsional) Panel tombol ipywidgets
Alternatif tombol untuk keputusan cepat. Jika widget bermasalah di Colab, abaikan sel ini —
API fungsional (`decide_ok()` dll) tetap jalan.

In [ ]:
try:
    import ipywidgets as W
    from IPython.display import display
    from google.colab import output as _co; _co.enable_custom_widget_manager()

    _name = W.Text(description="Nama:", placeholder="opsional")
    _mtgt = W.IntText(description="Merge→", value=-1)
    b_ok  = W.Button(description="OK", button_style="success")
    b_dis = W.Button(description="Discard", button_style="warning")
    b_mrg = W.Button(description="Merge", button_style="info")
    b_spl = W.Button(description="Split k=2", button_style="danger")
    b_nxt = W.Button(description="Skip→")
    _out = W.Output()
    def _wrap(fn):
        def h(_):
            with _out:
                _out.clear_output(wait=True); fn()
        return h
    b_ok.on_click(_wrap(lambda: decide_ok(_name.value or None)))
    b_dis.on_click(_wrap(lambda: decide_discard()))
    b_mrg.on_click(_wrap(lambda: decide_merge(_mtgt.value, _name.value or None)))
    b_spl.on_click(_wrap(lambda: decide_split(2)))
    b_nxt.on_click(_wrap(lambda: review_next()))
    display(W.HBox([b_ok, b_dis, b_mrg, b_spl, b_nxt]), W.HBox([_name, _mtgt]), _out)
    print("Panel siap. (Tetap bisa pakai API fungsional di sel bawah.)")
except Exception as e:
    print("ipywidgets tidak aktif:", e, "\n→ Pakai API fungsional: decide_ok() / decide_merge() / decide_split().")

## 8. MULAI REVIEW
Jalankan `review_next()` → lihat grid → panggil salah satu `decide_*`. Ulangi.
Gunakan `review_next(REVIEW_FIRST)` untuk membatasi ke 30 cluster terbesar dulu.

In [ ]:
review_next(REVIEW_FIRST)   # tampilkan cluster terbesar yg belum diputuskan (batch 30)

In [ ]:
# Contoh keputusan (jalankan SATU sesuai kondisi grid di atas, lalu grid berikutnya muncul):
# decide_ok()                 # cluster bersih, auto ID_<cid>
# decide_ok('Ahmad Fauzi')    # cluster bersih, beri nama
# decide_merge(12)            # cluster ini = orang yg sama dgn cluster 12
# decide_discard()            # bukan wajah / terlalu campur
# decide_split(2)             # ada 2 orang → lalu: assign_split({0:'Budi', 1:'Ani'})
#
# Bandingkan cluster lain tanpa memutuskan:  show_cluster(12)

## 9. Status progres
Berapa cluster sudah direview & cakupan wajah.

In [ ]:
def progress_summary():
    done = {int(k): v for k, v in progress.items()}
    tot_cl = len(review_queue)
    by = Counter(v["status"] for v in done.values())
    covered = 0
    for c in review_queue:
        r = done.get(c)
        if not r: continue
        if r["status"] in ("ok", "merge"): covered += sizes[c]
        elif r["status"] == "split":
            covered += sum(v is not None for v in r.get("split_assign", {}).values())
    first_done = sum(1 for c in review_queue[:REVIEW_FIRST] if c in done)
    print(f"Cluster direview : {len(done)}/{tot_cl}   (batch-30: {first_done}/{min(REVIEW_FIRST,tot_cl)})")
    print(f"Rincian status   : {dict(by)}")
    print(f"Wajah tercakup   : {covered} (dari {sum(sizes.values())} wajah ter-cluster)")
    nxt = _next_pending()
    print(f"Berikutnya       : cluster {nxt}" if nxt else "Semua selesai ✓")

progress_summary()

## 10. Finalisasi → `face_labels_verified.csv`
Konsolidasi semua keputusan menjadi label per wajah:
- **MERGE** → union-find atas `cluster_id`; komponen berbagi satu identitas.
- **SPLIT** → identitas per wajah dari `assign_split`.
- **Nama sama** (diketik manusia) → tergabung (join by name).
- **DISCARD** → `DISCARD`; **noise HDBSCAN** → `NOISE_HDBSCAN`; **belum direview** → `UNVERIFIED`.

Boleh dijalankan kapan saja (mis. setelah batch-30) untuk snapshot parsial.

In [ ]:
import pandas as pd

def finalize(save=True):
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b): parent[find(a)] = find(b)

    for k, r in progress.items():
        if r["status"] == "merge" and r.get("merge_with") is not None:
            union(int(k), int(r["merge_with"]))
    # nama manusia per komponen (kalau ada)
    comp_name = {}
    for k, r in progress.items():
        if r["status"] in ("ok", "merge") and r.get("true_id") and not str(r["true_id"]).startswith("ID_"):
            comp_name.setdefault(find(int(k)), r["true_id"])

    rows = []
    for gi in range(len(meta)):
        c = int(labels[gi]); m = meta[gi]
        if c < 0:
            tid, st = "NOISE_HDBSCAN", "noise"
        else:
            r = progress.get(str(c))
            if r is None:
                tid, st = "UNVERIFIED", "pending"
            elif r["status"] == "discard":
                tid, st = "DISCARD", "discard"
            elif r["status"] == "split":
                tid = r.get("split_assign", {}).get(str(gi))
                tid, st = (tid if tid else "DISCARD"), "split"
            else:  # ok / merge
                root = find(c); tid, st = comp_name.get(root, f"ID_{root:03d}"), r["status"]
        rows.append({"global_index": gi, "photo_path": m["photo_path"],
                     "bbox": m["bbox"], "proxy_cluster": c, "true_id": tid, "status": st})
    df = pd.DataFrame(rows)
    # id integer utk metrik hilir (hanya wajah terverifikasi & bukan discard)
    verified = ~df["true_id"].isin(["UNVERIFIED", "NOISE_HDBSCAN", "DISCARD"])
    df["identity_id"] = -1
    df.loc[verified, "identity_id"] = pd.factorize(df.loc[verified, "true_id"])[0]

    if save:
        df.to_csv(OUT_DIR / "face_labels_verified.csv", index=False)
        print("Tersimpan:", OUT_DIR / "face_labels_verified.csv")
    n_id = df.loc[verified, "true_id"].nunique()
    print(f"Wajah terverifikasi: {int(verified.sum())} / {len(df)}  →  {n_id} identitas unik")
    print("Distribusi status  :", dict(Counter(df["status"])))
    return df

df_labels = finalize()
df_labels.head(10)

## 11. Cara pakai ringkas
1. Jalankan sel **Setup → Config → Muat data → semua helper** (sekali).
2. `review_next(REVIEW_FIRST)` → grid cluster terbesar muncul.
3. Putuskan: `decide_ok()` / `decide_ok('Nama')` / `decide_merge(cid)` / `decide_split(k)` /
   `decide_discard()`. Grid berikutnya otomatis muncul.
4. Untuk SPLIT: `decide_split(2)` → lihat 2 sub-grup → `assign_split({0:'A', 1:'B'})`.
5. Berhenti kapan saja. Buka lagi notebook → jalankan sel setup → `review_next()` **lanjut
   otomatis** dari cluster berikutnya (progres tersimpan di Drive).
6. `progress_summary()` untuk cek kemajuan; `finalize()` untuk ekspor CSV (boleh parsial).

**Melanjutkan sisa cluster (di luar 30):** cukup `review_next()` **tanpa argumen** →
melanjutkan seluruh antrian sampai 144 cluster habis.

### Catatan untuk bab Metodologi
- Ground-truth = verifikasi visual manusia atas citra wajah (bukan geometri embedding).
- Seed = cluster HDBSCAN NB05; koreksi manusia via merge/split/discard.
- Keterbatasan: seed sekeluarga dengan metode yang dievaluasi → laporkan sebagai
  *human-verified cluster labels*, dan utamakan pelaporan ARI/NMI (memperhitungkan noise)
  di samping Purity.